Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [4]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(43800, 6)

In [10]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 1
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 6)
Dimensiones de Y: (43788, 1)


In [15]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 6)
Las dimensiones de testX son:  (8801, 12, 6)
Las dimensiones de valX son:  (4336, 12, 6)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

240/240 - 21s - 86ms/step - ia: 0.2313 - loss: 1.1224 - mae: 0.7976 - rmse: 1.0535 - smape: 1.5265 - val_ia: 0.2484 - val_loss: 0.9671 - val_mae: 0.7350 - val_rmse: 0.8737 - val_smape: 1.9485

Epoch 2/128                                           

240/240 - 9s - 35ms/step - ia: 0.2047 - loss: 1.0736 - mae: 0.7753 - rmse: 1.0306 - smape: 1.5562 - val_ia: 0.2495 - val_loss: 0.9652 - val_mae: 0.7355 - val_rmse: 0.8737 - val_smape: 1.9824

Epoch 3/128                                           

240/240 - 5s - 21ms/step - ia: 0.1826 - loss: 1.0428 - mae: 0.7656 - rmse: 1.0150 - smape: 1.5924 - val_ia: 0.2525 - val_loss: 0.9509 - val_mae: 0.7233 - val_rmse: 0.8625 - val_smape: 1.8196

Epoch 4/128                                           

240/240 - 5s - 21ms/step - ia: 0.1909 - loss: 0.9957 - mae: 0.7429 - rmse: 0.9913 - smape: 1.5741 - val_ia: 0.2813 - val_loss: 0.8757 - val_mae: 0.6916 - val_rmse: 0.8265 - val_smape: 1.6550

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

1916/1916 - 54s - 28ms/step - ia: 0.7792 - loss: 0.2039 - mae: 0.2809 - rmse: 0.3988 - smape: 0.6170 - val_ia: 0.6545 - val_loss: 0.0595 - val_mae: 0.1454 - val_rmse: 0.1990 - val_smape: 0.3828

Epoch 2/128                                                                         

1916/1916 - 44s - 23ms/step - ia: 0.8856 - loss: 0.0689 - mae: 0.1558 - rmse: 0.2337 - smape: 0.3883 - val_ia: 0.6546 - val_loss: 0.0629 - val_mae: 0.1483 - val_rmse: 0.2032 - val_smape: 0.3854

Epoch 3/128                                                                         

1916/1916 - 81s - 42ms/step - ia: 0.8902 - loss: 0.0670 - mae: 0.1504 - rmse: 0.2287 - smape: 0.3757 - val_ia: 0.6929 - val_loss: 0.0581 - val_mae: 0.1348 - val_rmse: 0.1891 - val_smape: 0.3513

Epoch 4/128                                                                         

1916/1916 - 84s - 44ms/step - ia: 0.8926 - loss: 0.0638 - mae: 0.1459 -

...

Epoch 1/128                                                                         

1916/1916 - 29s - 15ms/step - ia: 0.5040 - loss: 0.6468 - mae: 0.5754 - rmse: 0.7501 - smape: 1.1181 - val_ia: 0.3989 - val_loss: 0.2008 - val_mae: 0.2964 - val_rmse: 0.3607 - val_smape: 0.6611

Epoch 2/128                                                                         

1916/1916 - 23s - 12ms/step - ia: 0.7215 - loss: 0.2631 - mae: 0.3625 - rmse: 0.4875 - smape: 0.7569 - val_ia: 0.4260 - val_loss: 0.1583 - val_mae: 0.2813 - val_rmse: 0.3408 - val_smape: 0.6595

Epoch 3/128                                                                         

1916/1916 - 23s - 12ms/step - ia: 0.7582 - loss: 0.2074 - mae: 0.3162 - rmse: 0.4304 - smape: 0.6826 - val_ia: 0.4931 - val_loss: 0.1190 - val_mae: 0.2264 - val_rmse: 0.2856 - val_smape: 0.5423

Epoch 4/128                                                                         

1916/1916 - 22s - 11ms/step - ia: 0.7828 - loss: 0.1744 - mae: 0.2882 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                           

3832/3832 - 31s - 8ms/step - ia: 0.2355 - loss: 1.1633 - mae: 0.8428 - rmse: 1.0255 - smape: 1.6544 - val_ia: 0.1763 - val_loss: 1.0118 - val_mae: 0.7988 - val_rmse: 0.8357 - val_smape: 1.7365

Epoch 2/128                                                                           

3832/3832 - 23s - 6ms/step - ia: 0.2382 - loss: 1.1262 - mae: 0.8294 - rmse: 1.0087 - smape: 1.6579 - val_ia: 0.1777 - val_loss: 0.9913 - val_mae: 0.7891 - val_rmse: 0.8260 - val_smape: 1.7435

Epoch 3/128                                                                           

3832/3832 - 22s - 6ms/step - ia: 0.2395 - loss: 1.0935 - mae: 0.8183 - rmse: 0.9915 - smape: 1.6640 - val_ia: 0.1788 - val_loss: 0.9729 - val_mae: 0.7800 - val_rmse: 0.8169 - val_smape: 1.7499

Epoch 4/128                                                                           

3832/3832 - 23s - 6ms/step - ia: 0.2387 - loss: 1.0741 - mae: 0.80

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                           

240/240 - 35s - 147ms/step - ia: 0.4550 - loss: 0.6698 - mae: 0.5883 - rmse: 0.7984 - smape: 1.1950 - val_ia: 0.6910 - val_loss: 0.2118 - val_mae: 0.3172 - val_rmse: 0.4199 - val_smape: 0.7070

Epoch 2/128                                                                           

240/240 - 7s - 28ms/step - ia: 0.7491 - loss: 0.2476 - mae: 0.3542 - rmse: 0.4927 - smape: 0.7374 - val_ia: 0.7600 - val_loss: 0.1423 - val_mae: 0.2420 - val_rmse: 0.3436 - val_smape: 0.5678

Epoch 3/128                                                                           

240/240 - 10s - 40ms/step - ia: 0.7806 - loss: 0.2001 - mae: 0.3112 - rmse: 0.4429 - smape: 0.6732 - val_ia: 0.7860 - val_loss: 0.1186 - val_mae: 0.2159 - val_rmse: 0.3146 - val_smape: 0.5205

Epoch 4/128                                                                           

240/240 - 11s - 44ms/step - ia: 0.8000 - loss: 0.1700 - mae: 0.2850 -

In [22]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
